Implementation a toy-example Question Answering system. Sample of the learned model to generate answers when using questions as prompts. Report of best results and discussions of findings in experiments.   


In [ ]:
# download the tiny Trivial Q/A dataset from Google drive
#
!gdown --folder https://drive.google.com/drive/folders/1cLjPppGCbjL6w31F1l70Q8qtzL-8pRUb?usp=share_link 2> /dev/null

Processing file 1aJNW3CKrwduxL35tBIya6TbaoNWWQc8Y answers.txt
Processing file 11KhbiY23sdpJkqzkIQQe4jLycPns6MdI prompts.txt
Processing file 1iUbTcBJEHSbA9R9OfMwKr0Lvt5S4Hbeh train.txt


In [ ]:
# display the first few lines of training text, containing question and answer pairs

!cat tinyQA/train.txt | head -n 10

what was pierce brosnan's first outing as 007 [ goldeneye ]
the 02 arena is in which london borough [ greenwich ]
who wrote the 1956 novel '101 dalmatians' [ dodie smith ]
which band's first top ten single was the 10538 overture in 1972 [ electric light orchestra ]
the 1999 film 10 things i hate about you is based on which shakespeare play [ the taming of the shrew ]
the film `10 things i hate about you` is based on which shakespeare play [ the taming of the shrew ]
the film '10 things i hate about you', was inspired by which of shakespeare's plays [ the taming of the shrew ]
who directed the 2010 film 127 hours [ danny boyle ]
ciara had a hit with 1,2 step featuring which other artist [ missy elliot ]
which film director won the oscar for best picture for the film 12 years a slave in 2013 [ steve mcqueen ]


In [ ]:
# display the six questions as prompts

!cat tinyQA/prompts.txt

which football club did alan sugar own [
 'the black and gold' is a nickname of which american football team [ 
alex band and aaron kamin make up which band [
the aberdare mountains are in which african country [
which notable leader won the 2009 nobel peace prize [
who commanded the confederate army of northern virginia during the american civil war [ 


In [ ]:
# display the questions and answers for all prompts

!cat tinyQA/answers.txt

which football club did alan sugar own [ tottenham hotspur f.c. ]
'the black and gold' is a nickname of which american football team [ pittsburgh steelers ]
alex band and aaron kamin make up which band [ the calling ]
the aberdare mountains are in which african country [ kenya ]
which notable leader won the 2009 nobel peace prize [ barack obama ]
who commanded the confederate army of northern virginia during the american civil war [ robert e. lee ]


In [ ]:
# -*- coding: utf-8 -*-
"""
## Assignment Three - Question 4 (EECS3404 W26)
### Transformers for Question Answering
"""

# STEP 1: Download the tinyQA dataset

!gdown --folder https://drive.google.com/drive/folders/1cLjPppGCbjL6w31F1l70Q8qtzL-8pRUb?usp=share_link 2> /dev/null

print("=== TRAINING DATA (first 10 lines) ===")
!cat tinyQA/train.txt | head -n 10

print("\n=== PROMPTS (questions) ===")
!cat tinyQA/prompts.txt

print("\n=== ANSWERS (correct answers) ===")
!cat tinyQA/answers.txt

# STEP 2: Define the GPT model (same as Project 7)

import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

class config():
    def __init__(self, batch_size=10, n_layer=12, n_head=12, n_embd=768,
                 block_size=1024, vocab_size=50304, causal=True, device='cpu'):
        assert n_embd % n_head == 0
        self.batch_size = batch_size
        self.n_embd = n_embd
        self.block_size = block_size
        self.n_head = n_head
        self.causal = causal
        self.device = device
        self.n_layer = n_layer
        self.vocab_size = vocab_size

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Key, query, value projections for all heads packed into one matrix
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # Causal (lower-triangular) mask to prevent attending to future tokens
        self.register_buffer("mask", torch.tril(torch.ones(config.block_size, config.block_size))
                             .transpose(0, 1).view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()  # batch size, sequence length, embedding dim
        # Split the projection into queries, keys, and values
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, C // self.n_head, self.n_head).transpose(1, 3)  # (B, nh, hs, T)
        q = q.view(B, T, C // self.n_head, self.n_head).transpose(1, 3)  # (B, nh, hs, T)
        v = v.view(B, T, C // self.n_head, self.n_head).transpose(1, 3)  # (B, nh, hs, T)
        # Scaled dot-product attention with causal mask
        att = (q.transpose(-2, -1) @ k) * (1.0 / math.sqrt(q.size(-2)))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-2)  # column-wise softmax
        y = v @ att
        # Re-assemble all head outputs side by side
        y = y.transpose(1, 3).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.relu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    """Single transformer block: LayerNorm -> Attention -> LayerNorm -> MLP, with residual connections."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm([config.n_embd])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm([config.n_embd])
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))  # attention sub-layer with residual
        x = x + self.mlp(self.ln_2(x))   # feed-forward sub-layer with residual
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),   # token embeddings
            wpe  = nn.Embedding(config.block_size, config.n_embd),   # position embeddings
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),  # transformer blocks
            ln_f = nn.LayerNorm([config.n_embd]),                     # final layer norm
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying: share token embedding and output projection weights
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        # Special scaled init for residual projections (per GPT-2 paper)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        print("number of parameters: %.2fM" % (self.get_num_params() / 1e6,))

    def get_num_params(self, non_embedding=True):
        """Return the number of parameters (excluding position embeddings by default)."""
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0)  # position indices
        # Token + position embeddings
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = tok_emb + pos_emb
        # Pass through all transformer blocks
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            # Training: compute loss over all positions
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # Inference: only compute logits for the last token (more efficient)
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Auto-regressively generate new tokens given a prompt."""
        for _ in range(max_new_tokens):
            # Crop context to block_size if it has grown too long
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            # Scale logits by temperature (lower = more deterministic)
            logits = logits[:, -1, :] / temperature
            # Optionally keep only the top-k most likely tokens
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            # Sample the next token from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# STEP 3: Load and prepare the tinyQA dataset

train_txt_file = 'tinyQA/train.txt'

with open(train_txt_file, 'r') as f:
    data = f.read()
print(f"Length of dataset in characters: {len(data):,}")

# Build a character-level vocabulary from the training text
chars = sorted(list(set(data)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Character-to-index and index-to-character mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    """Convert a string to a list of integer token IDs."""
    return [stoi[c] for c in s if c in stoi]

def decode(l):
    """Convert a list of token IDs back to a string."""
    return ''.join([itos[i] for i in l])

# 90/10 train/validation split
n = len(data)
train_data = data[:int(n * 0.9)]
val_data   = data[int(n * 0.9):]

train_ids = encode(train_data)
val_ids   = encode(val_data)
print(f"Train tokens: {len(train_ids):,} | Val tokens: {len(val_ids):,}")

# STEP 4: Hyperparameters (best config for tinyQA)

# --- Model size ---
# Small model chosen deliberately: tinyQA is a tiny dataset,
# a large model would overfit immediately.
n_layer   = 4    # number of transformer layers
n_head    = 4    # number of attention heads per layer
head_size = 32   # dimensionality of each attention head
n_embd    = n_head * head_size  # total embedding dim = 128

# --- Training ---
learning_rate = 3e-3   # higher than default to converge faster on small data
max_iters     = 5000   # total training steps
batch_size    = 16     # mini-batch size
block_size    = 64     # context window (sufficient for short Q&A pairs)

# --- AdamW optimizer ---
beta1 = 0.9
beta2 = 0.95

# --- Learning rate schedule (cosine decay with linear warmup) ---
decay_lr       = True
warmup_iters   = 200   # linear warmup over first 200 steps
lr_decay_iters = 5000  # decay completes at max_iters
min_lr         = 3e-4  # floor = learning_rate / 10
grad_clip      = 1.0   # gradient clipping threshold

# --- Evaluation ---
eval_interval = 500   # evaluate every N steps
eval_iters    = 50    # number of batches to average for loss estimate

# --- System ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = 'bfloat16'

print(f"Using device: {device}")
print(f"Model: n_layer={n_layer}, n_head={n_head}, n_embd={n_embd}")

torch.manual_seed(42)

# STEP 5: Data loader

data_tr  = np.array(train_ids, dtype=np.uint16)
data_val = np.array(val_ids,   dtype=np.uint16)

def get_batch(split):
    """Sample a random mini-batch of (input, target) pairs from train or val set."""
    data = data_tr if split == 'train' else data_val
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

# STEP 6: Initialize the model

conf = config(
    batch_size=batch_size,
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    block_size=block_size,
    vocab_size=vocab_size,
    device=device
)

model = GPT(conf).to(device)

# GradScaler only active for float16; bfloat16 does not need it
scaler    = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(beta1, beta2))

@torch.no_grad()
def estimate_loss():
    """Estimate average loss on train and val splits without updating weights."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def get_lr(it):
    """Cosine learning rate schedule with linear warmup."""
    if it < warmup_iters:                        # 1) linear warmup
        return learning_rate * it / warmup_iters
    if it > lr_decay_iters:                      # 2) after decay: hold at min_lr
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # 3) cosine decay
    return min_lr + coeff * (learning_rate - min_lr)

# STEP 7: Training loop

print("\n========== TRAINING ==========")
X, Y = get_batch('train')
t0 = time.time()
iter_num = 0

while True:
    # Update learning rate according to schedule
    lr = get_lr(iter_num) if decay_lr else learning_rate
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Forward pass
    _, loss = model(X, Y)
    # Pre-fetch next batch while GPU runs backward
    X, Y = get_batch('train')

    # Backward pass with gradient scaling (for fp16 stability)
    scaler.scale(loss).backward()
    # Clip gradients to prevent exploding gradients
    if grad_clip != 0.0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)  # free gradient memory immediately

    # Periodic evaluation
    if iter_num % eval_interval == 0:
        t1 = time.time()
        losses = estimate_loss()
        print(f"step {iter_num:4d}: train loss {losses['train']:.4f}, "
              f"val loss {losses['val']:.4f}  ({t1-t0:.1f}s)")
        t0 = t1

    iter_num += 1
    if iter_num > max_iters:
        break

print("Training complete!")

# STEP 8: Sampling — evaluate the 6 prompt questions

num_samples    = 5    # 5 attempts per question (as required by the assignment)
max_new_tokens = 50   # keep answers short for Q&A
temperature    = 0.8  # slightly below 1.0 for more focused outputs
top_k          = 10   # restrict sampling to the top-10 most likely tokens

torch.manual_seed(42)
model.eval()
model.to(device)

def sample_prompt(prompt, n=5):
    """Generate n candidate answers for a given prompt string."""
    start_ids = encode(prompt)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]
    results = []
    for k in range(n):
        y = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        decoded = decode(y[0].tolist())
        # Extract only the first line after the prompt (the predicted answer)
        after_prompt = decoded[len(prompt):]
        first_line = after_prompt.split('\n')[0].strip()
        results.append(first_line)
    return results

print("\n========== QUESTION ANSWERING RESULTS ==========")
print("(5 samples per question — correct if ANY sample matches)\n")

# Load the six test prompts and their correct answers
with open('tinyQA/prompts.txt', 'r') as f:
    prompts = [line.rstrip('\n') for line in f if line.strip()]

with open('tinyQA/answers.txt', 'r') as f:
    answers_raw = f.read()

print("CORRECT ANSWERS:")
print(answers_raw)
print("=" * 50)

# Run sampling for each question and display all 5 candidate answers
for i, prompt in enumerate(prompts):
    print(f"\nQ{i+1}: {prompt}")
    samples = sample_prompt(prompt, n=num_samples)
    for j, s in enumerate(samples):
        print(f"  Sample {j+1}: {s}")
    print()

Processing file 1aJNW3CKrwduxL35tBIya6TbaoNWWQc8Y answers.txt
Processing file 11KhbiY23sdpJkqzkIQQe4jLycPns6MdI prompts.txt
Processing file 1iUbTcBJEHSbA9R9OfMwKr0Lvt5S4Hbeh train.txt
=== TRAINING DATA (first 10 lines) ===
what was pierce brosnan's first outing as 007 [ goldeneye ]
the 02 arena is in which london borough [ greenwich ]
who wrote the 1956 novel '101 dalmatians' [ dodie smith ]
which band's first top ten single was the 10538 overture in 1972 [ electric light orchestra ]
the 1999 film 10 things i hate about you is based on which shakespeare play [ the taming of the shrew ]
the film `10 things i hate about you` is based on which shakespeare play [ the taming of the shrew ]
the film '10 things i hate about you', was inspired by which of shakespeare's plays [ the taming of the shrew ]
who directed the 2010 film 127 hours [ danny boyle ]
ciara had a hit with 1,2 step featuring which other artist [ missy elliot ]
which film director won the oscar for best picture for the film 1

/tmp/ipykernel_4282/794350478.py:292: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


step    0: train loss 4.0563, val loss 4.0565  (0.1s)
step  500: train loss 2.0056, val loss 2.1425  (53.2s)
step 1000: train loss 1.7196, val loss 1.9212  (52.4s)
step 1500: train loss 1.5080, val loss 1.8879  (54.7s)
step 2000: train loss 1.3396, val loss 1.8893  (54.1s)
step 2500: train loss 1.1634, val loss 1.9170  (54.2s)
step 3000: train loss 0.9948, val loss 2.0610  (54.1s)
step 3500: train loss 0.8543, val loss 2.2391  (54.0s)
step 4000: train loss 0.7096, val loss 2.4622  (53.3s)
step 4500: train loss 0.6157, val loss 2.6352  (54.5s)
step 5000: train loss 0.5637, val loss 2.8265  (53.6s)
Training complete!

========== QUESTION ANSWERING RESULTS ==========
(5 samples per question — correct if ANY sample matches)

CORRECT ANSWERS:
which football club did alan sugar own [ tottenham hotspur f.c. ]
'the black and gold' is a nickname of which american football team [ pittsburgh steelers ]
alex band and aaron kamin make up which band [ the calling ]
the aberdare mountains are in whic

# Experiment Findings



## most important obervations

As it can be seen, the model only got 1 question parcially right (question 1 tottenham) so it is obvious that the model is clearly struggling

### overfitting

It can be seen that this model is overfitting really badly. While the train loss keeps improving overtime from 4.05 to 0.56, the val loss starts rising after step 1500 from 4.05 to 2.82

This means that the model is memorizing the training Q&A pairs intead of accually generalize patterns since these both losses should decrease together

### small dataset

100k characters is extremely small for a language model, it dosent have anough data to acctually learn. It just memorize the training set wivh causes the overfitting explained earlier.

### positive outcomes

Even though the responses itself were not precise, it can be evidenced with the outputs that the model learned some structure since in Q3 all answers stared with "the" (from the response "the calling"), Q4 produce real African countries and Q5 produced "al gore" which is a reasonable answer for a nobel price question


##  Possible improve changes

We could stop training the model around step 1500 where val loss is the lowest since the model is actually worst at step 5000. We could also try a smaller model or add dropout to reduce overfitting. Finally we could repeat the training data by duplicating it so the model sees Q&A pairs more ofter without overfitting as fast.